In [1]:
import gymnasium as gym
from stable_baselines3.common.evaluation import evaluate_policy
from Approach_env import SRC_approach
import numpy as np
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_checker import check_env
from RL_algo.td3_BC import TD3_BC
from RL_algo.DemoHerReplayBuffer import DemoHerReplayBuffer
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.noise import NormalActionNoise
import time
import pickle
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn

seed = 10
set_random_seed(seed)

with open('/home/surgic-ai/SurgicAI/RL/Env_info/Approach_noise_env_info', 'rb') as file:
    env_info = pickle.load(file)
    
step_size= np.array(env_info["step_size"], dtype=np.float32)
threshold = np.array(env_info["threshold"], dtype=np.float32)
episode_steps = int(env_info["max_timestep"])

trans_step = 1.0e-3
angle_step = np.deg2rad(3)
jaw_step = 0.05
step_size = np.array([trans_step, trans_step, trans_step, angle_step, angle_step, angle_step, jaw_step], dtype=np.float32)

threshold = np.array([0.3, np.deg2rad(70)], dtype=np.float32)
episode_steps = 1000

# Register and make the env
gym.envs.register(id="TD3_HER_BC", entry_point=SRC_approach, max_episode_steps=episode_steps)
env = gym.make("TD3_HER_BC", render_mode="human",reward_type = "dense",max_episode_step=episode_steps,seed = seed, step_size=step_size,threshold=threshold)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


max episode length is 1000
step size is [0.001      0.001      0.001      0.05235988 0.05235988 0.05235988
 0.05      ]
Set random seed
reward type is dense
Translation threshold: 0.30000001192092896, angle threshold: 1.2217304706573486
kinematics loaded from /home/surgic-ai/SurgicAI/RL/utils/kinematics/config/kinematic/psm_420006.json  and  /home/surgic-ai/SurgicAI/RL/utils/kinematics/config/tool/LARGE_NEEDLE_DRIVER_420006.json
kinematics loaded from /home/surgic-ai/SurgicAI/RL/utils/kinematics/config/kinematic/psm_420006.json  and  /home/surgic-ai/SurgicAI/RL/utils/kinematics/config/tool/LARGE_NEEDLE_DRIVER_420006.json
Initialized!!!


In [2]:
# Check the environment
#check_env(env.unwrapped)

In [3]:
with open('/home/surgic-ai/SurgicAI/RL/Approach_td3/base_env/all_episodes_merged_again.pkl', 'rb') as file:
    episode_transitions = pickle.load(file)

In [4]:
episode_transitions = None

In [5]:
# Set up the hyperparameters
goal_selection_strategy = "future"
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=5e-2 * np.ones(n_actions))


model = TD3_BC(
    "MultiInputPolicy",
    env,
    learning_rate=3e-4,
    learning_starts=600,
    tau = 0.005,
    gamma = 0.995,
    batch_size=512,
    action_noise=action_noise,
    replay_buffer_class=DemoHerReplayBuffer,
    train_freq = (3, "episode"),
    policy_kwargs = dict(net_arch=dict(pi=[256, 256, 256], qf=[256, 256, 256])),
    # Parameters for HER
    replay_buffer_kwargs=dict(
        demo_transitions=episode_transitions, 
        demo_sample_ratio=0.2,
        n_sampled_goal=4,
        goal_selection_strategy=goal_selection_strategy,
    ),
    verbose=1,
    tensorboard_log="./Approach/TD3_HER_BC",
    episode_transitions=episode_transitions,
    BC_coeff=0.3,
    demo_ratio=0.1,
)
model_path = "/home/surgic-ai/SurgicAI/RL/Approach/TD3_HER_BC/dense/seed_10/base_env/final_model.zip"
model = TD3_BC.load(model_path,env=env)#

checkpoint_callback = CheckpointCallback(save_freq=5000, save_path='/home/surgic-ai/SurgicAI/RL/Approach/TD3_HER_BC/dense/seed_10/', name_prefix='rl_model')

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/surgic-ai/SurgicAI/RL/RL_algo/DemoHerReplayBuffer.py:38: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  observations[key] = th.tensor(observations[key], dtype=th.float32, device=self.device)


In [ ]:
# Train the model
model.learn(total_timesteps=int(300000), progress_bar=True,callback=checkpoint_callback,reset_num_timesteps=False)

reset!!!
Logging to learning_data/Approach/TD3_HER_BC/dense/seed_10/tensorboard/TD3_0


/usr/lib/python3/dist-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/surgic-ai/SurgicAI/.venv/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:245: 
UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: 
<class 'numpy.ndarray'>
  logger.warn(

Matched degree is 17.709921006860867, distance_trans = 0.2992125718470282, distances_angle = 53.574362660245704

Attach the needle to the gripper

reset!!!

In [ ]:
# Save the model
model.save("/home/surgic-ai/SurgicAI/RL/Approach/TD3_BC_noise_dense/rl_model_final")

In [ ]:
# Predict the action with model
obs,info = env.reset()
print(obs)
for i in range(90000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.01)
    if terminated or truncated:
        obs, info = env.reset()